# Control 1: Is the CGP routing weight just a myeloid indicator?

Reviewer E7Cp, follow-up comment (Nov 19, 2026):

> "Compare the correlations in Table 9 with a simple myeloid indicator, e.g. the
> expression of LYZ or CD14, or just the myeloid cell-type label. If these simple
> indicators give similar correlations with the same signatures, then the CGP
> routing weight is mostly acting as a myeloid indicator and does not add extra
> information."

For every one of the 4,440 gene modules we compute, across the 13,654 ifnb cells:

- `r_w`: correlation of module activity with the **CGP expert routing weight**
 (seeds 42 / 43 / 44 of the full model);
- `r_LYZ`, `r_CD14`: correlation with the normalized expression of LYZ / CD14;
- `r_label`: correlation with a binary myeloid label (DC, CD14 Mono, CD16 Mono);
- `r_partial`: partial correlation of the CGP weight with module activity after
 residualizing both on the myeloid label (i.e. the signal the routing weight
 carries *beyond* cell type);
- `r_partial_LYZ`: same, controlling for LYZ expression.

Table 9 of the revised paper reports the top entries of the `r_w` ranking
(seed 42). All paths are relative; run this notebook from the `ICLR_review/`
directory (kernel: the `deepfusion` conda environment).

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import anndata

ROOT = Path("..")            # repository root (one level above ICLR_review/)
IFNB = ROOT / "ifnb"
EXP02 = Path("exp02_expert_masking")
OUT = Path("exp08_e7cp_controls")
OUT.mkdir(exist_ok=True)

MYELOID = ["DC", "CD14 Mono", "CD16 Mono"]   # pDC / Mk / Eryth excluded (ambiguous lineage)
LYMPHOID = ["CD4 Naive T", "CD4 Memory T", "CD8 T", "T activated",
            "B", "B Activated", "NK"]

## Load data

- Module activity matrix (13,654 cells x 4,440 modules), rebuilt from the
 preprocessed expression data and the module definitions — the same matrix
 behind Table 9.
- Per-cell CGP routing weights for three seeds, stored in the integrated
 object at training time (not reconstructed).
- Normalized expression of LYZ and CD14.

In [2]:
mm = torch.load(EXP02 / "module_matrix_4440_rebuilt.pt", map_location="cpu").numpy()
vm = pickle.load(open(IFNB / "enhanced_results/DeepFusion/cache/valid_modules_cache_dd0.5.pkl", "rb"))
module_names = list(vm.keys())

ad = anndata.read_h5ad(IFNB / "enhanced_results/results/step5_add_DeepFusion.h5ad")
labels = ad.obs["seurat_annotations"].astype(str).values
assert mm.shape[0] == ad.n_obs

var_names = list(ad.var_names.astype(str))
def gene_expr(g):
    x = ad.layers["normalized"][:, var_names.index(g)]
    return (x.toarray() if sp.issparse(x) else np.asarray(x)).ravel()

lyz, cd14 = gene_expr("LYZ"), gene_expr("CD14")
myel = np.isin(labels, MYELOID).astype(float)

W = {s: np.asarray(ad.obsm[k])[:, 0] for s, k in
     [("seed42", "X_expert_weights_40_std"),
      ("seed43", "X_expert_weights_40_std_s43"),
      ("seed44", "X_expert_weights_40_std_s44")]}   # column 0 = CGP expert

id2name = pd.read_csv(IFNB / "SC_data/c2.all.v2026.1.Hs.symbols_ID_to_category_name_filtered_parent.csv")
id2name = dict(zip(id2name["labels"].astype(str), id2name["name"]))
gmap = json.load(open(IFNB / "SC_data/module_groups_v2026.json"))
groups = np.array([gmap[k] for k in module_names])

def pretty(mname):
    return id2name.get(mname.replace("Module_", ""), mname)

print(f"cells={ad.n_obs}, modules={len(module_names)}, "
      f"myeloid cells={int(myel.sum())} ({100*myel.mean():.1f}%)")

cells=13654, modules=4440, myeloid cells=5724 (41.9%)


## Correlations of every module with the CGP weight and the three indicators

In [3]:
def corr(a, b):
    if a.std() < 1e-12 or b.std() < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def partial_corr(x, y, c):
    """Partial correlation of x and y given a single control variable c."""
    C = np.column_stack([np.ones_like(c), c])
    rx = x - C @ np.linalg.lstsq(C, x, rcond=None)[0]
    ry = y - C @ np.linalg.lstsq(C, y, rcond=None)[0]
    return corr(rx, ry)

rows = []
for i, mn in enumerate(module_names):
    a = mm[:, i]
    rows.append({
        "module": mn, "name": pretty(mn), "group": groups[i],
        "r_w_seed42": corr(a, W["seed42"]),
        "r_w_seed43": corr(a, W["seed43"]),
        "r_w_seed44": corr(a, W["seed44"]),
        "r_LYZ": corr(a, lyz),
        "r_CD14": corr(a, cd14),
        "r_label": corr(a, myel),
        "r_partial_label": partial_corr(a, W["seed42"], myel),
        "r_partial_LYZ": partial_corr(a, W["seed42"], lyz),
    })
df = pd.DataFrame(rows)
df.to_csv(OUT / "control1_all_modules_correlations.csv", index=False)
print(df.shape)

(4440, 11)


## Table A — the Table 9 top signatures, against the simple indicators

Top 15 CGP modules by correlation with the CGP routing weight (seed 42),
with the same module's correlation with LYZ, CD14, the myeloid label, and the
partial correlation beyond the label. If the CGP weight were just a myeloid
indicator, the partial correlations would collapse to ~0 and the indicator
columns would match `r_w_seed42`.

In [4]:
cgp = df[df.group == "CGP"]
top15 = cgp.nlargest(15, "r_w_seed42")
cols = ["name", "r_w_seed42", "r_LYZ", "r_CD14", "r_label",
        "r_partial_label", "r_partial_LYZ"]
print(top15[cols].round(3).to_string(index=False))

                                                                   name  r_w_seed42  r_LYZ  r_CD14  r_label  r_partial_label  r_partial_LYZ
                         FULCHER_INFLAMMATORY_RESPONSE_LECTIN_VS_LPS_DN       0.873  0.625   0.267    0.767            0.652          0.781
                                       VERHAAK_AML_WITH_NPM1_MUTATED_UP       0.866  0.697   0.463    0.847            0.516          0.741
                                              GAL_LEUKEMIC_STEM_CELL_DN       0.859  0.718   0.456    0.826            0.521          0.720
                           RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP       0.845  0.690   0.457    0.797            0.523          0.704
                                             RUTELLA_RESPONSE_TO_HGF_UP       0.835  0.672   0.357    0.756            0.554          0.691
                                  RUTELLA_RESPONSE_TO_CSF2RB_AND_IL4_DN       0.834  0.672   0.452    0.783            0.510          0.690
                    

## Table B — does any indicator beat the routing weight? (all CGP modules)

In [5]:
summary = {
    "max r (CGP modules)": [cgp.r_w_seed42.max(), cgp.r_LYZ.max(),
                            cgp.r_CD14.max(), cgp.r_label.max(),
                            cgp.r_partial_label.max()],
    "mean of top-10 r": [cgp.r_w_seed42.nlargest(10).mean(),
                         cgp.r_LYZ.nlargest(10).mean(),
                         cgp.r_CD14.nlargest(10).mean(),
                         cgp.r_label.nlargest(10).mean(),
                         cgp.r_partial_label.nlargest(10).mean()],
}
summary = pd.DataFrame(summary, index=["CGP weight (s42)", "LYZ expr",
                                       "CD14 expr", "myeloid label",
                                       "CGP weight | label (partial)"])
print(summary.round(3))

best_indicator = df[["r_LYZ", "r_CD14", "r_label"]].max(axis=1)
frac = (df.r_w_seed42 > best_indicator).mean()
print(f"\nFraction of all modules where the CGP weight correlates more strongly "
      f"than the best simple indicator: {frac:.3f}")

                              max r (CGP modules)  mean of top-10 r
CGP weight (s42)                            0.873             0.842
LYZ expr                                    0.718             0.679
CD14 expr                                   0.613             0.537
myeloid label                               0.847             0.794
CGP weight | label (partial)                0.652             0.607

Fraction of all modules where the CGP weight correlates more strongly than the best simple indicator: 0.852


## Table C — top correlates of the simple indicators

What the myeloid indicators themselves correlate with most strongly. If the
CGP weight were redundant with them, these lists would be the same as Table A.

In [6]:
for ind in ["r_LYZ", "r_CD14", "r_label"]:
    print(f"== top-10 modules by {ind} ==")
    print(df.nlargest(10, ind)[["name", "group", ind]].round(3).to_string(index=False))
    print()

== top-10 modules by r_LYZ ==
                                            name group  r_LYZ
                       GAL_LEUKEMIC_STEM_CELL_DN   CGP  0.718
                VERHAAK_AML_WITH_NPM1_MUTATED_UP   CGP  0.697
    RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP   CGP  0.690
                         DELYS_THYROID_CANCER_UP   CGP  0.686
                            LIAN_LIPA_TARGETS_3M   CGP  0.679
            LENAOUR_DENDRITIC_CELL_MATURATION_DN   CGP  0.675
                      RUTELLA_RESPONSE_TO_HGF_UP   CGP  0.672
           RUTELLA_RESPONSE_TO_CSF2RB_AND_IL4_DN   CGP  0.672
SMIRNOV_CIRCULATING_ENDOTHELIOCYTES_IN_CANCER_UP   CGP  0.659
                       MARKEY_RB1_CHRONIC_LOF_DN   CGP  0.645

== top-10 modules by r_CD14 ==
                                                                     name    group  r_CD14
            REACTOME_RESPIRATORY_SYNCYTIAL_VIRUS_RSV_ATTACHMENT_AND_ENTRY REACTOME   0.865
   REACTOME_REGULATION_OF_TBK1_IKK_IKBKE_MEDIATED_ACTIVATION_OF_IRF3_IRF7 

## Three-seed consistency of the Table 9 signatures

In [7]:
print(top15[["name", "r_w_seed42", "r_w_seed43", "r_w_seed44"]].round(3).to_string(index=False))

                                                                   name  r_w_seed42  r_w_seed43  r_w_seed44
                         FULCHER_INFLAMMATORY_RESPONSE_LECTIN_VS_LPS_DN       0.873       0.848       0.870
                                       VERHAAK_AML_WITH_NPM1_MUTATED_UP       0.866       0.882       0.895
                                              GAL_LEUKEMIC_STEM_CELL_DN       0.859       0.868       0.884
                           RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP       0.845       0.862       0.870
                                             RUTELLA_RESPONSE_TO_HGF_UP       0.835       0.841       0.845
                                  RUTELLA_RESPONSE_TO_CSF2RB_AND_IL4_DN       0.834       0.843       0.857
                                                DELYS_THYROID_CANCER_UP       0.830       0.853       0.860
                                              MARKEY_RB1_CHRONIC_LOF_DN       0.826       0.831       0.852
                            